# VHH HEFT predictions

Closed-form **$W^\pm HH$** and **$ZHH$** at LO / NNLO. Run from the **repo root**.

| § | What |
|---|------|
| 1 | Configuration |
| 2 | Spot check at one $\kappa$ |
| 3 | `scan_and_save()` → `Results/Points/` |
| 4–5 | Plots (optional) |
| 6 | Wilson tables |


## Setup

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path(".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from vhh_predict import (
    CHANNELS,
    TABLE_ENERGIES_TEV,
    build_channel_tables,
    format_prediction,
    latex_wilson_tables_for_process,
    load_analysis,
    plots_dir,
    resolve_scan_axis,
    scan_and_save,
    scan_axes,
    tables_dir,
)
from vhh_predict.plots import (
    X_LABELS,
    plot_enhancement_only,
    plot_kfactor_only,
    plot_sigma_lo_only,
    plot_sigma_nnlo_and_enhancement_nnlo,
    plot_sigma_nnlo_and_kfactor,
    plot_sigma_only,
)
from vhh_predict.tables import KAPPA_PLAIN, ZHH_TABLE_GROUPS

PLOTS_DIR = plots_dir()
TABLES_DIR = tables_dir()
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)


## 1. Configuration

| Variable | Meaning |
|----------|---------|
| `PROCESS` | `WplusHH`, `WminusHH`, `ZHH` |
| `ENERGY_TEV` | `13.6` or `14.0` |
| `KAPPA` | W$\pm$: `(κ_λ, κ_W, κ_{2W})` — ZHH: `(κ_λ, κ_Z, κ_{2Z}, κ_t)` |
| `COMPARE_SIMULATION` | Compare to MadGraph in §2 spot check only |
| `SAVE_SCAN_POINTS` | Write scan JSON to `Results/Points/` (§3) |
| `SAVE_PLOTS` | Write PNGs to `Results/Plots/` (§4–5) |
| `SIGMA_INSET` | Zoom inset on σ_NNLO panels |


In [ ]:
PROCESS = "ZHH"
ENERGY_TEV = 14.0
KAPPA = (3.0, 1.0, 1.0, 1.0)  # ZHH: include κ_t

COMPARE_SIMULATION = True
UNCERTAINTIES_AS_PERCENT = True
SAVE_SCAN_POINTS = True
SAVE_PLOTS = True
SIGMA_INSET = True

analysis = load_analysis(PROCESS, ENERGY_TEV)
nn_label = analysis.nnlo_label
FIXED_KAPPA = KAPPA

print(f"{PROCESS} @ {ENERGY_TEV} TeV  |  scan axes: {', '.join(scan_axes(PROCESS))}")


## 2. Spot check

In [ ]:
prediction = format_prediction(
    analysis, KAPPA,
    as_percent=UNCERTAINTIES_AS_PERCENT,
    compare_simulation=COMPARE_SIMULATION,
)
print(prediction)


## 3. Scan (and optionally save)

One `scan_and_save()` call — results in `scan_data` for §4–5. Set `SAVE_SCAN_POINTS=True` to also write `Results/Points/{Process}_{energy}TeV_{axis}.json` (slim columns only; uncertainty bands stay in memory for plots).


In [ ]:
scan_axis = "kappa_t"
scan_vmin, scan_vmax = 0.85, 1.15
scan_n_points = 400

scan_data, scan_points_file = scan_and_save(
    analysis,
    scan_axis,
    vmin=scan_vmin,
    vmax=scan_vmax,
    fixed_kappa=FIXED_KAPPA,
    n_points=scan_n_points,
    uncertainties=True,
    save=SAVE_SCAN_POINTS,
)
if SAVE_SCAN_POINTS:
    print(f"Saved {scan_points_file}")


## 4. Single-panel plots

Uses `scan_data` from §3. Run §3 first.

In [ ]:
scan_idx, scan_x_key = resolve_scan_axis(PROCESS, scan_axis)
scan_axis_label = X_LABELS[scan_x_key]
kappa_names = [r"\lambda", "Z", r"2Z", "t"] if PROCESS == "ZHH" else [r"\lambda", "W", r"2W"]
n_kappa = 4 if PROCESS == "ZHH" else 3
fixed_title = ", ".join(
    rf"$\kappa_{{{kappa_names[i]}}}={FIXED_KAPPA[i]:g}$"
    for i in range(n_kappa) if i != scan_idx
)
scan_title_base = f"{PROCESS} @ {ENERGY_TEV} TeV — {scan_axis_label} ({fixed_title})"
prefix = f"{PROCESS}_{ENERGY_TEV}TeV_{scan_x_key}"

plot_sigma_only(scan_data, title=scan_title_base, nnlo_label=nn_label,
    sigma_inset=SIGMA_INSET,
    output=PLOTS_DIR / f"{prefix}_sigma_nnlo.png", save=SAVE_PLOTS)
plt.show()

plot_sigma_lo_only(scan_data, title=scan_title_base,
    output=PLOTS_DIR / f"{prefix}_sigma_lo.png", save=SAVE_PLOTS)
plt.show()

plot_kfactor_only(scan_data, title=scan_title_base, nnlo_label=nn_label,
    output=PLOTS_DIR / f"{prefix}_K.png", save=SAVE_PLOTS)
plt.show()

for order, tag in (("LO", "sigmaSM_LO"), (nn_label, f"sigmaSM_{nn_label}")):
    plot_enhancement_only(scan_data, order=order, show_uncertainty=False,
        title=f"{scan_title_base} — σ_HEFT/σ_SM ({order})", nnlo_label=nn_label,
        output=PLOTS_DIR / f"{prefix}_{tag}.png", save=SAVE_PLOTS)
    plt.show()


## 5. Two-panel plots

In [ ]:
plot_sigma_nnlo_and_kfactor(scan_data, title=scan_title_base, nnlo_label=nn_label,
    sigma_inset=SIGMA_INSET,
    output=PLOTS_DIR / f"{prefix}_sigma_nnlo_K.png", save=SAVE_PLOTS)
plt.show()

plot_sigma_nnlo_and_enhancement_nnlo(scan_data, title=scan_title_base, nnlo_label=nn_label,
    sigma_inset=SIGMA_INSET, show_enhancement_uncertainty=False,
    output=PLOTS_DIR / f"{prefix}_sigma_nnlo_sigmaSM_{nn_label}.png", save=SAVE_PLOTS)
plt.show()


## 6. Kappa tables

In [ ]:
latex_path = TABLES_DIR / "Kappa_tables.tex"
latex_blocks = []
for process in CHANNELS:
    display(Markdown(f"## {process}"))
    tables = build_channel_tables(process, energies_tev=TABLE_ENERGIES_TEV)
    if process == "ZHH":
        for axes in ZHH_TABLE_GROUPS:
            key = "_".join(axes)
            display(Markdown(f"### {', '.join(KAPPA_PLAIN[a] for a in axes)}"))
            display(tables[key])
    else:
        display(tables)
    tex = latex_wilson_tables_for_process(process, energies_tev=TABLE_ENERGIES_TEV)
    latex_blocks.append(tex)
    display(Markdown("**LaTeX**"))
    print(tex)
latex_path.write_text("\n\n".join(latex_blocks), encoding="utf-8")
print(f"Saved {latex_path}")
